<a href="https://colab.research.google.com/github/Jopat2409/com3610_notebooks/blob/main/LUKE_trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Accelerate

In [3]:
%%capture
%pip install git+https://github.com/huggingface/accelerate
%pip install datasets
%pip install evaluate
%pip install -U datasets

Setup and complete imports

In [4]:
import os
import math
import random
import logging
import argparse
import unicodedata
from pathlib import Path

import torch
import datasets
import evaluate
import transformers

import pandas as pd

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from huggingface_hub import Repository, create_repo
from transformers.utils.versions import require_version
from accelerate import Accelerator, DistributedDataParallelKwargs
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict

from transformers import (
    AdamW,
    LukeConfig,
    LukeForEntitySpanClassification,
    LukeTokenizer,
    SchedulerType,
    default_data_collator,
    get_scheduler,
    set_seed,
)

logger = logging.getLogger(__name__)

logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
)

require_version("datasets>=1.8.0", "To fix: pip install -r examples/pytorch/token-classification/requirements.txt")

#### Utils

In [5]:
def is_punctuation(char):
    cp = ord(char)
    if (cp >= 33 and cp <= 47) or (cp >= 58 and cp <= 64) or (cp >= 91 and cp <= 96) or (cp >= 123 and cp <= 126):
        return True
    cat = unicodedata.category(char)
    if cat.startswith("P"):
        return True
    return False

#### Dataset Loading

In [20]:
def load_data(train_path: str, valid_path: str):
    data_files = {"train": train_path, "validation": valid_path}
    return load_dataset("csv" if (ext := train_path.split(".")[-1]) == "txt" else ext, data_files=data_files, delimiter='\s+')

def _load_conll_split(path: str):
    dat = pd.read_csv(path, sep="\s+", skip_blank_lines=False, usecols=[0, 3], names=["TOKENS", "TAGS"], keep_default_na=False, engine="python")
    dat["SENTENCE_BOUNDARY"] = (dat["TOKENS"] == "").cumsum()
    dat = dat[~dat['TOKENS'].isin(['', "-DOCSTART-"])]

    sentences = (
        dat.groupby("SENTENCE_BOUNDARY")["TOKENS"]
    ).apply(lambda x: x.tolist()).reset_index().rename(columns={"TOKENS": "tokens"})

    tags = (
        dat.groupby("SENTENCE_BOUNDARY")["TAGS"]
    ).apply(lambda x: x.tolist()).reset_index().rename(columns={"TAGS": "ner_tags"})

    assert len(sentences) == len(tags), "Invalid number of tags and sentences"
    processed_data = tags.merge(sentences, on="SENTENCE_BOUNDARY", how="left").drop(columns=["SENTENCE_BOUNDARY"])

    dataset = Dataset.from_pandas(processed_data)

    print(dataset.features)

    return dataset

def load_conll_data(directory: str):
    train = _load_conll_split(os.path.join(directory, "train.txt"))
    valid = _load_conll_split(os.path.join(directory, "dev.txt"))

    return DatasetDict({"train": train, "validation": valid})

In [7]:
def get_label_list(labels):
    unique_labels = set()
    for label in labels:
        unique_labels = unique_labels | set(label)
    label_list = list(unique_labels)
    label_list.sort()
    return label_list

In [8]:
def clamp_to_max_length(subword_lengths, sentence_end, tokens, max_length = 128) -> tuple:
    context_end = sentence_end

    if sum(subword_lengths) > 128 - 2:
        cur_length = sum(subword_lengths[:context_end])
        idx = context_end - 1

        while cur_length > 128 - 2:
            cur_length -= subword_lengths[idx]
            context_end -= 1
            idx -= 1

    return tokens[:context_end], subword_lengths[:context_end]

In [29]:
def compute_entity_spans_for_luke(examples, tokenizer):
    texts, all_entity_spans, all_labels_entity_spans, all_original_entity_spans = [], [], [], []

    for labels, tokens, sentence_boundaries in zip(examples["ner_tags"], examples["tokens"], examples["sentence_boundaries"]):
        subword_lengths = [len(tokenizer.tokenize(token)) for token in tokens]
        total_subword_length = sum(subword_lengths)

        sentence_words, sentence_subword_lengths = clamp_to_max_length(subword_lengths, sentence_boundaries[1], tokens)

        text = ""
        word_start_char_positions = []
        word_end_char_positions = []
        labels_positions = {}

        for word, label in zip(sentence_words, labels):
            if word[0] == "'" or (len(word) == 1 and is_punctuation(word)):
                text = text.rstrip()

            word_start_char_positions.append(len(text))
            text += word
            word_end_char_positions.append(len(text))
            text += " "
            labels_positions[(word_start_char_positions[-1], word_end_char_positions[-1])] = label

        text = text.rstrip()
        texts.append(text)
        entity_spans = []
        labels_entity_spans = []
        original_entity_spans = []

        for word_start in range(len(sentence_words)):
            for word_end in range(word_start, len(sentence_words)):
                if (
                    sum(sentence_subword_lengths[word_start:word_end]) <= tokenizer.max_mention_length
                    and len(entity_spans) < tokenizer.max_entity_length
                ):
                    entity_spans.append((word_start_char_positions[word_start], word_end_char_positions[word_end]))
                    original_entity_spans.append((word_start, word_end + 1))
                    if (
                        word_start_char_positions[word_start],
                        word_end_char_positions[word_end],
                    ) in labels_positions:
                        labels_entity_spans.append(
                            labels_positions[
                                (word_start_char_positions[word_start], word_end_char_positions[word_end])
                            ]
                        )
                    else:
                        labels_entity_spans.append(0)

        all_entity_spans.append(entity_spans)
        all_labels_entity_spans.append(labels_entity_spans)
        all_original_entity_spans.append(original_entity_spans)

    examples["entity_spans"] = all_entity_spans
    examples["text"] = texts
    examples["labels_entity_spans"] = all_labels_entity_spans
    examples["original_entity_spans"] = all_original_entity_spans

    return examples

In [58]:
def compute_entity_spans_for_luke_2(examples, tokenizer):
    all_entity_spans = []
    texts = []
    all_labels_entity_spans = []
    all_original_entity_spans = []

    for labels, tokens, sentence_boundaries in zip(
        examples["ner_tags"], examples["tokens"], examples["sentence_boundaries"]
    ):
        subword_lengths = [len(tokenizer.tokenize(token)) for token in tokens]
        total_subword_length = sum(subword_lengths)
        _, context_end = sentence_boundaries

        if total_subword_length > 126:
            cur_length = sum(subword_lengths[:context_end])
            idx = context_end - 1

            while cur_length > 126:
                cur_length -= subword_lengths[idx]
                context_end -= 1
                idx -= 1

        text = ""
        sentence_words = tokens[:context_end]
        sentence_subword_lengths = subword_lengths[:context_end]
        word_start_char_positions = []
        word_end_char_positions = []
        labels_positions = {}

        for word, label in zip(sentence_words, labels):
            if word[0] == "'" or (len(word) == 1 and is_punctuation(word)):
                text = text.rstrip()

            word_start_char_positions.append(len(text))
            text += word
            word_end_char_positions.append(len(text))
            text += " "
            labels_positions[(word_start_char_positions[-1], word_end_char_positions[-1])] = label

        text = text.rstrip()
        texts.append(text)
        entity_spans = []
        labels_entity_spans = []
        original_entity_spans = []

        for word_start in range(len(sentence_words)):
            for word_end in range(word_start, len(sentence_words)):
                if (
                    sum(sentence_subword_lengths[word_start:word_end]) <= tokenizer.max_mention_length
                    and len(entity_spans) < tokenizer.max_entity_length
                ):
                    entity_spans.append((word_start_char_positions[word_start], word_end_char_positions[word_end]))
                    original_entity_spans.append((word_start, word_end + 1))
                    if (
                        word_start_char_positions[word_start],
                        word_end_char_positions[word_end],
                    ) in labels_positions:
                        labels_entity_spans.append(
                            labels_positions[
                                (word_start_char_positions[word_start], word_end_char_positions[word_end])
                            ]
                        )
                    else:
                        labels_entity_spans.append(0)

        all_entity_spans.append(entity_spans)
        all_labels_entity_spans.append(labels_entity_spans)
        all_original_entity_spans.append(original_entity_spans)

    examples["entity_spans"] = all_entity_spans
    examples["text"] = texts
    examples["labels_entity_spans"] = all_labels_entity_spans
    examples["original_entity_spans"] = all_original_entity_spans

    return examples

In [67]:
def tokenize_and_align_labels(examples, tokenizer):
    entity_spans = [[tuple(e) for e in ex] for ex in examples["entity_spans"]]

    tokenized_inputs = tokenizer(
        examples["text"],
        entity_spans=entity_spans,
        max_length=128,
        padding=False,
        truncation=True,
    )

    for to, _from in [("labels", "labels_entity_spans"), ("original_entity_spans", "original_entity_spans"), ("ner_tags", "ner_tags")]:
      tokenized_inputs[to] = [ex[:tokenizer.max_entity_length] for ex in examples[_from]]

    return tokenized_inputs

In [12]:
def compute_sentence_boundaries_for_luke(examples):
        sentence_boundaries = []

        for tokens in examples["tokens"]:
            sentence_boundaries.append([0, len(tokens)])

        examples["sentence_boundaries"] = sentence_boundaries

        return examples

In [71]:
def process_dataset(accelerator: Accelerator, dataset: DatasetDict, tokenizer):
        raw_datasets = dataset.map(
            compute_sentence_boundaries_for_luke,
            batched=True,
            desc="Adding sentence boundaries"
        )
        raw_datasets = raw_datasets.map(
            compute_entity_spans_for_luke_2,
            batched=True,
            desc="Adding sentence spans",
            fn_kwargs={"tokenizer": tokenizer}
        )

        processed_raw_datasets = raw_datasets.map(
            tokenize_and_align_labels,
            batched=True,
            remove_columns=raw_datasets["train"].column_names,
            desc="Running tokenizer on dataset",
            fn_kwargs={"tokenizer": tokenizer}
        )

        return processed_raw_datasets

In [ ]:
def create_dataloaders(processed_dataset, accelerator: Accelerator, tokenizer, per_device_train_batch_size=32):

    if accelerator.mixed_precision == "fp8":
        pad_to_multiple_of = 16
    elif accelerator.mixed_precision != "no":
        pad_to_multiple_of = 8
    else:
        pad_to_multiple_of = None

    data_collator = DataCollatorForLukeTokenClassification(tokenizer, pad_to_multiple_of=pad_to_multiple_of)

    train_dataloader = DataLoader(processed_dataset["train"], shuffle=True, collate_fn=data_collator, batch_size=per_device_train_batch_size)
    eval_dataloader = DataLoader(processed_dataset["validation"], collate_fn=data_collator, batch_size=per_device_train_batch_size)

    return train_dataloader, eval_dataloader

#### Model Loading

In [56]:
%%capture
def load_transformers_components(name: str, num_labels: int, max_entity_length: int, max_mention_length: int) -> tuple[LukeTokenizer, LukeForEntitySpanClassification]:
    config = LukeConfig.from_pretrained(name, num_labels=num_labels)

    tokenizer = LukeTokenizer.from_pretrained(
        LUKE_BASE,
        use_fast=False,
        task="entity_span_classification",
        max_entity_length=max_entity_length,
        max_mention_length=max_mention_length,
    )

    model = LukeForEntitySpanClassification.from_pretrained(
        name,
        from_tf=bool(".ckpt" in name),
        config=config,
    )

    model.resize_token_embeddings(len(tokenizer))

    return tokenizer, model

#### Logging Setup

In [15]:
def setup_logging(accelerator: Accelerator):
    logger.info(accelerator.state)

    logger.setLevel(logging.INFO if accelerator.is_local_main_process else logging.ERROR)

    if accelerator.is_local_main_process:
        datasets.utils.logging.set_verbosity_warning()
        transformers.utils.logging.set_verbosity_info()
    else:
        datasets.utils.logging.set_verbosity_error()
        transformers.utils.logging.set_verbosity_error()


## Main Training Script

In [16]:
LUKE_BASE = "studio-ousia/luke-base"
MAX_ENTITY_LENGTH = 32
MAX_MENTION_LENGTH = 30

In [76]:
def main():

    handler = DistributedDataParallelKwargs(find_unused_parameters=True)
    accelerator = Accelerator(kwargs_handlers=[handler])

    setup_logging(accelerator)

    #dataset = load_conll_data("./cleanconll")
    dataset = load_dataset("conll2003", trust_remote_code=True)

    print(dataset["train"][0])

    unique_labels = get_label_list(dataset["train"]["ner_tags"])
    num_labels = len(unique_labels)

    #b_to_i = [unique_labels.index(label.replace("B-", "I-")) if label.startswith("B-") else idx for idx, label in enumerate(unique_labels)]

    tokenizer, model = load_transformers_components(LUKE_BASE, num_labels, MAX_ENTITY_LENGTH, MAX_MENTION_LENGTH)
    processed_dataset = process_dataset(accelerator, dataset, tokenizer)

    print(processed_dataset["train"].features)

    train_dataloader, eval_dataloader = create_dataloaders(processed_dataset, accelerator, tokenizer)


In [77]:
main()

INFO:__main__:Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cpu

Mixed precision type: no



{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--studio-ousia--luke-base/snapshots/7438924defd9f3c2018d63c16073bf4bcb6a70aa/config.json
Model config LukeConfig {
  "_name_or_path": "studio-ousia/luke-base",
  "architectures": [
    "LukeForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bert_model_name": "roberta-base",
  "bos_token_id": 0,
  "classifier_dropout": null,
  "entity_emb_size": 256,
  "entity_vocab_size": 500000,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "L

{'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None), 'original_entity_spans': Sequence(feature=Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), length=-1, id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'entity_ids': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'entity_position_ids': Sequence(feature=Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), length=-1, id=None), 'entity_start_positions': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'entity_end_positions': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'entity_attention_mask': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'labels': Sequence(feature=Value(dtype='int64'

NameError: name 'create_dataloaders' is not defined